In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key = os.getenv("GROQ_API_KEY"),
    base_url = "https://api.groq.com/openai/v1"
)

In [7]:
def llm(instructions, user_prompt, model ="openai/gpt-oss-20b" ):
    message_history = [
        {"role" : "developer", "content": instructions},
        {"role" : "user", "content": user_prompt}
    ]
    
    response = openai_client.responses.create(
        # model="qwen/qwen3.8-27b",
        model=model,
        input=message_history
    )

    return response.output_text


In [8]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

question = "I just discovered the course. Can I join now?"

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

answer = llm(prompt)

TypeError: llm() missing 1 required positional argument: 'user_prompt'

In [1]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

documents = []
url_prefix = "https://datatalks.club/faq"
for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1400

In [2]:
from minsearch import Index

index = Index(
    text_fields=["questions", "sections", "answer"],
    keyword_fields=["course"]
)
index.fit(documents)

question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question":2.0, "section":0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)
search_results


#wrapping it in a function
def search(question, course = "llm-zoomcamp"):
    boost_dict = {"question":2.0, "section":0.5}
    filter_dict = {"course" : course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [16]:
[doc["question"] for doc in search_results]

['Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Do I have to use OpenAI, or can I use a different provider?',
 'Can I run the course locally instead of Codespaces?',
 'OpenAI: Do I have to subscribe and pay for Open AI API for this course?',
 'Why do the matrix and for-loop versions of vector search give slightly different results?']

# Building the Prompt

In [3]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question: 
{question}

Context: 
{context}
"""

def build_context(search_results):
    lines =[]

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

prompt = build_prompt(question, search_results)
print(prompt)

Question: 
I just discovered the course. Can I join now?

Context: 
General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

Module 1: RAG
Q: Do I have to use OpenAI, or can I use a different provider?
A: If the provider used in the course isn't available or is blocked in your region (or you simply prefer another one), you can use any other LLM provider — the course isn't tied to OpenAI. Just switch to something else:

- Hosted, OpenAI-compatible providers — e.g. Groq, OpenRouter, DeepSeek, Gemini, Z.ai, Mistral. The course code uses the OpenAI client, so you usually only need to change the `base_url`, the API key, and the model name.
- Open models via Hugging

In [11]:
def rag(query, model = "openai/gpt-oss-20b"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model)
    return answer

# rag("I Just discovered the course. Can I join now?")
rag("How do I run Olama locally?")

"I don't know."

# Agents

In [12]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [13]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [16]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]
response = openai_client.responses.create(
    model="openai/gpt-oss-20b",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseReasoningItem(id='resp_01m3t3d1gfe0nba3vg87wkgv4p', summary=[], type='reasoning', content=[Content(text='The user asks: "I just discovered the course. Can I join it?" They want to know if they can enroll in the course. We need to check if the user is a student, and if the course is available, if the student has prerequisites, if enrollment is possible.\n\nWe have no direct data on the course or student status. The system likely has a function to search FAQ. Let\'s search for "enrollment" or "join course" or similar.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"can I join the course enrollment policy"}', call_id='fc_ecf54988-7b06-4810-89d9-b4c9ee4c177b', name='search', type='function_call', id='fc_ecf54988-7b06-4810-89d9-b4c9ee4c177b', async_=None, caller=None, namespace=None, status='completed')]

In [22]:
import json

call = response.output[1]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)
result_json

'[\n  {\n    "id": "85384a18e5",\n    "course": "llm-zoomcamp",\n    "section": "Module 1: RAG",\n    "question": "OpenAI: Do I have to subscribe and pay for Open AI API for this course?",\n    "answer": "No, you don\'t have to pay for this service in order to complete the course homeworks. You can use free or low-cost alternatives listed in the course GitHub repo.\\n\\nSee the course list of [OpenAI API alternatives](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/awesome-llms.md#openai-api-alternatives)."\n  },\n  {\n    "id": "2fed4b176c",\n    "course": "llm-zoomcamp",\n    "section": "Capstone Project",\n    "question": "Can I use AI tools like ChatGPT, Claude, or Cursor to write code for the capstone project?",\n    "answer": "Yes. AI tools are allowed for homework, the project, and peer reviews.\\n\\nWhat we ask is that you understand what you submit. If a peer reviewer asks why\\nyou picked a particular chunking strategy, retrieval setup, or orchestrator, you\\nshould b

In [27]:
messages.extend(response.output)
messages.append(
    {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json
    }
)

messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseReasoningItem(id='resp_01m3t3d1gfe0nba3vg87wkgv4p', summary=[], type='reasoning', content=[Content(text='The user asks: "I just discovered the course. Can I join it?" They want to know if they can enroll in the course. We need to check if the user is a student, and if the course is available, if the student has prerequisites, if enrollment is possible.\n\nWe have no direct data on the course or student status. The system likely has a function to search FAQ. Let\'s search for "enrollment" or "join course" or similar.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"can I join the course enrollment policy"}', call_id='fc_ecf54988-7b06-4810-89d9-b4c9ee4c177b', name='search', type='function_call', id='fc_ecf54988-7b06-4810-89d9-b4c9ee4c177b', async_=None, caller=None, namespace=None, status='completed'),
 ResponseReasoningItem(id='resp

In [ ]:
response = openai_client.responses.create(
    model="openai/gpt-oss-20b",
    input=messages,
    tools=[search_tool],   
)
response.output_text

''

In [30]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }


In [31]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="openai/gpt-oss-20b",
    input=messages,
    tools=[search_tool],   
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"just discovered the course. can i join it?"}


In [ ]:
def agent_loop(question, instructions = instructions, model="openai/gpt-oss-20b") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer


agent_loop(question=question)

In [40]:
# agent_loop(question="How do I run Ollama locally?")
# agent_loop(question="How do I run Olama locally?")
agent_loop(question="I just discovered the course. Can I still join it?", model="openai/gpt-oss-120b")

iteration #1...
function_call: search {"query":"Can I still join the course after it has started enrollment deadline"}
iteration #2...
function_call: search {"query":"Can I still join the course after it has started enrollment"}
iteration #3...
function_call: search {"query":"late enrollment can I still join the course after it started"}
iteration #4...
function_call: search {"query":"Can I still join the course after it has started? enrollment open"}
iteration #5...
function_call: search {"query":"Can I join the course after it has started? enrollment closed"}
iteration #6...
function_call: search {"query":"Can I still join the course after it has started? FAQ"}
iteration #7...
function_call: search {"query":"late enrollment join after start LLM Zoomcamp FAQ"}
iteration #8...
function_call: search {"query":"Can I still join the LLM Zoomcamp after it started enrollment open"}
iteration #9...
function_call: search {"query":"enrollment open can I still join LLM Zoomcamp"}
iteration #10..

'**Yes – you can still join the LLM\u202fZoomcamp even if the cohort has already started.**  \n\nHere’s how it works:\n\n| What you need to do | Where to do it | What to expect |\n|---------------------|----------------|----------------|\n| **Enroll / create an account** | The\u202f[course‑management platform](https://courses.datatalks.club/llm-zoomcamp-2026/) (or the sign‑up link you received) | Once you’re logged in you’ll see the current cohort’s syllabus, homework forms, and peer‑review windows. |\n| **Access the learning material** | • GitHub repo\u202f<https://github.com/DataTalksClub/llm-zoomcamp>  <br>• Video links listed at the top of each lesson folder  <br>• The “Zoomcamp docs” page (<https://datatalks.club/docs/courses/llm-zoomcamp/>) | All videos, notebooks, and reading material are publicly available, so you can start whenever you like. |\n| **Follow the weekly workflow** | Follow the steps outlined in the FAQ **“How should I start the course and follow the weekly workflo

In [43]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions=instructions,question= "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
ASSISTANT:
It looks like your question isn’t about course content or logistics. I’m only able to answer questions related to the course or its scheduling. If you have any other questions about the class, assignments, or deadlines, feel free to let me know!


'It looks like your question isn’t about course content or logistics. I’m only able to answer questions related to the course or its scheduling. If you have any other questions about the class, assignments, or deadlines, feel free to let me know!'